## Use LDA to inspect life history strategy separation for sparse, dense, NSM latents. 
---
Compares three representations of the same specimens side by side: the 28 sparse landmarks, the
dense (population) correspondences, and the NSM latent codes. All three come from the same
`all_vtk_files` order, so one `specimens` table (species / family / trait / color / marker, joined
from `lizard_species_list.csv`) drives every plot.

*Last edited 13 Sep 2026 by K. Wolcott*

In [2]:
# Imports, paths, and config

import os, re, json, ast, torch
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap.umap_ as umap

from NSM.plotting import (load_mrk_json, plot_life_history_legend, plot_family_color_legend,
                          sort_key, plotly_color)
from NSM.morphometrics import *   # gm_prcomp, two_d_array, mshape, centroid_size, ...

# ---------------------------------------------------------------- TO DO: edit
RUN          = "run_v72"                      # training attempt directory
ATLAS_RUN    = "2026_07-15_13_06_22/"         # atlas/builder run that produced the landmark sets
ATLAS_ROOT = Path("/home/k.wolcott/UFL Dropbox/Katherine Wolcott/neural_shape_models/final_dataset_aug26/atlas/")
CKPT         = "2500"                         # latent code checkpoint to analyze
# -----------------------------------------------------------------------------

cwd       = Path.cwd()
base_wd   = cwd.parent
train_dir = base_wd / RUN
os.chdir(train_dir)
print(f"Working directory: {os.getcwd()}")

ATLAS_DIR       = ATLAS_ROOT / ATLAS_RUN / "atlas"
SPARSE_LM_DIR   = ATLAS_ROOT / ATLAS_RUN / "alignedLMs"
DENSE_LM_DIR    = ATLAS_ROOT / ATLAS_RUN / "population_correspondences"
SPARSE_MEAN_FN  = ATLAS_DIR / "atlas_sparse_landmarks.mrk.json"
DENSE_MEAN_FN   = ATLAS_DIR / "atlas_dense_correspondences.mrk.json"
MEAN_MESH_FN    = ATLAS_DIR / "atlas_model.ply"

OUT_DIR = Path("pca_tsne_umap_results")
OUT_DIR.mkdir(exist_ok=True)
print(f"Outputs will be written to: {OUT_DIR.resolve()}")

for p in [ATLAS_DIR, SPARSE_LM_DIR, DENSE_LM_DIR, SPARSE_MEAN_FN, DENSE_MEAN_FN, MEAN_MESH_FN]:
    print(("  OK   " if p.exists() else "  MISS ") + str(p))

# Load config and filenames
with open("model_params_config.json") as f:
    cfg = json.load(f)
print(f"\033[92mLoaded config from model_params_config.json\033[0m")

# Load NSM latent codes
CKPT_PATH = f"latent_codes/{CKPT}.pth"
latent_ckpt = torch.load(CKPT_PATH, map_location="cpu")
codes = latent_ckpt["latent_codes"]["weight"].detach().cpu().numpy()
print(f"Latent codes: {codes.shape}")

# Get filenames
all_vtk_files = [os.path.basename(f) for f in cfg["list_mesh_paths"]]
print(f"{len(all_vtk_files)} meshes listed in config")

# Check filename length against latent codes length
assert len(codes) == len(all_vtk_files), (
    f"{len(codes)} latent codes but {len(all_vtk_files)} meshes in config -- order/count mismatch")

Working directory: /home/k.wolcott/NSM/nsm/run_v72
Outputs will be written to: /home/k.wolcott/NSM/nsm/run_v72/pca_tsne_umap_results
  OK   /home/k.wolcott/UFL Dropbox/Katherine Wolcott/neural_shape_models/final_dataset_aug26/atlas/2026_07-15_13_06_22/atlas
  OK   /home/k.wolcott/UFL Dropbox/Katherine Wolcott/neural_shape_models/final_dataset_aug26/atlas/2026_07-15_13_06_22/alignedLMs
  OK   /home/k.wolcott/UFL Dropbox/Katherine Wolcott/neural_shape_models/final_dataset_aug26/atlas/2026_07-15_13_06_22/population_correspondences
  OK   /home/k.wolcott/UFL Dropbox/Katherine Wolcott/neural_shape_models/final_dataset_aug26/atlas/2026_07-15_13_06_22/atlas/atlas_sparse_landmarks.mrk.json
  OK   /home/k.wolcott/UFL Dropbox/Katherine Wolcott/neural_shape_models/final_dataset_aug26/atlas/2026_07-15_13_06_22/atlas/atlas_dense_correspondences.mrk.json
  OK   /home/k.wolcott/UFL Dropbox/Katherine Wolcott/neural_shape_models/final_dataset_aug26/atlas/2026_07-15_13_06_22/atlas/atlas_model.ply
Loaded

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


### Specimen metadata (species / family / trait / color / marker)

In [ ]:
# Build specimen metadata table -- one row per mesh, in the same order as `codes`

SPECIES_CSV = "../lizard_species_list.csv"
pat = re.compile(r"^(?P<species>[\w\s\-]+)[\-_ ]+[\w\d]+[\-_ ]+(?P<vertebra>[CTL]?\d+)", re.IGNORECASE)
parsed = [pat.match(f) for f in all_vtk_files]
specimens = pd.DataFrame({"mesh":        all_vtk_files,
                          "specimen_id": [m.group("species") if m else None for m in parsed],
                          "vertebra":    [m.group("vertebra").upper() if m else None for m in parsed]})
print(f"Parsed {specimens['specimen_id'].notna().sum()} / {len(specimens)} filenames")

sdf = pd.read_csv(SPECIES_CSV)
sdf["marker"] = sdf["marker"].astype(str).str.strip().str.strip("'" + chr(34))
sdf["color"]  = sdf["color"].apply(ast.literal_eval)
specimens = specimens.merge(sdf, left_on="specimen_id", right_on="specimen", how="left")

unmatched = specimens.loc[specimens["family"].isna(), "specimen_id"].dropna().unique()
print(f"{specimens['family'].notna().sum()} / {len(specimens)} specimens matched to {SPECIES_CSV}")
if len(unmatched):
    print(f"\033[33mUnmatched specimen IDs ({len(unmatched)}):\033[0m {sorted(unmatched)[:10]}")

REGION_NAMES = {"C": "CERVICAL", "T": "THORACIC", "L": "LUMBAR"}
specimens["region"] = specimens["vertebra"].str[0].map(REGION_NAMES)

print("Specimens dataframe head:\n", specimens.head())

# One color per broad clade (matches plot_family_color_legend elsewhere) -- computed once and
# reused both for the inline PNG legends below and the standalone legend cell at the end.
family_base_colors = (specimens.drop_duplicates("broad_taxon_for_plotting")
                     .set_index("broad_taxon_for_plotting")["color"].to_dict())

Parsed 2316 / 2316 filenames
2316 / 2316 specimens matched to ../lizard_species_list.csv
Specimens dataframe head:
                                      mesh                   specimen_id  \
0  agamidae_agama_atra_uf180711_01-c3.vtk  agamidae_agama_atra_uf180711   
1  agamidae_agama_atra_uf180711_02-c4.vtk  agamidae_agama_atra_uf180711   
2  agamidae_agama_atra_uf180711_03-c5.vtk  agamidae_agama_atra_uf180711   
3  agamidae_agama_atra_uf180711_04-c6.vtk  agamidae_agama_atra_uf180711   
4  agamidae_agama_atra_uf180711_05-c7.vtk  agamidae_agama_atra_uf180711   

  vertebra                      specimen broad_taxon_for_plotting    family  \
0       C3  agamidae_agama_atra_uf180711                  iguania  Agamidae   
1       C4  agamidae_agama_atra_uf180711                  iguania  Agamidae   
2       C5  agamidae_agama_atra_uf180711                  iguania  Agamidae   
3       C6  agamidae_agama_atra_uf180711                  iguania  Agamidae   
4       C7  agamidae_agama_atra_uf1807

In [ ]:
# Define functions

import seaborn as sns
from itertools import combinations
from matplotlib.colors import LinearSegmentedColormap
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from statsmodels.multivariate.manova import MANOVA
from statsmodels.stats.multitest import multipletests
from plotly.subplots import make_subplots
from sklearn.model_selection import (StratifiedKFold, StratifiedGroupKFold, cross_val_score,
                                     cross_val_predict, permutation_test_score)
from sklearn.metrics import make_scorer, balanced_accuracy_score, recall_score

# One color per trait, derived from that trait's marker 
markers = ['P', '+', 's', 'd', 'X', 'o', '2']
colors = [(0.65, 0.69, 0.12),   # pea soup
        (0.84, 0.65, 0.23),   # saffron
        (0.72, 0.44, 0.22),   # mud
        (0.36, 0.557, 0.68),  # powder blue
        (0.10, 0.51, 0.40),   # deep blue
        (0.60, 0.50, 0.46),   # slate
        (0, 0, 0)]            # black
marker_to_color = dict(zip(markers, colors))

# Match against traits in species_lis.csv
trait_marker = specimens.drop_duplicates("trait").set_index("trait")["marker"]
trait_colors = {t: marker_to_color.get(m, (0.5, 0.5, 0.5))
                for t, m in trait_marker.items() if pd.notna(t)}
specimens["trait_color"] = specimens["trait"].map(trait_colors)
unmapped = [t for t, m in trait_marker.items() if pd.notna(t) and m not in marker_to_color]
if unmapped:
    print(f"\033[33mTraits using a marker not in marker_to_color (defaulting to grey): {unmapped}\033[0m")
n_traits = len(trait_colors)
print(f"{n_traits} traits: {sorted(trait_colors)}")

def _trait_legend_traces():
    """One legend-only swatch per trait, for the static PNG export (see pc_pair_grid)."""
    return [go.Scatter(x=[None], y=[None], mode="markers",
                       marker=dict(size=20, color=plotly_color(col), symbol="circle"),
                       name=trait.upper(), legendgroup=trait, showlegend=True)
           for trait, col in trait_colors.items()]


def _family_legend_traces():
    """One legend-only swatch per broad clade, for the static PNG export (see pc_pair_grid)."""
    return [go.Scatter(x=[None], y=[None], mode="markers",
                       marker=dict(size=20, color=plotly_color(col), symbol="circle"),
                       name=fam.upper(), legendgroup=fam, showlegend=True)
           for fam, col in family_base_colors.items() if isinstance(fam, str)]

def get_gradient_cmap(color, n=256, white_level=0.8):
    light = tuple(c + (1 - c) * white_level for c in color)
    return LinearSegmentedColormap.from_list("gradient_cmap", [light, color], N=n)

def n_pcs_for_variance(cum, threshold):
    if threshold >= 1.0:
        return len(cum)
    return int(np.searchsorted(cum, threshold) + 1)

def fmt_p(val, significant=False):
    if pd.isna(val):
        return "NaN"
    if val == 0:
        s = "< 1e-300"
    else:
        exp = int(np.floor(np.log10(abs(val))))
        s = f"{val / 10**exp:.2f}e{exp:+03d}"
    return s + ("*" if significant else "")

# LDA + MANOVA + KDE functions 
def run_lda_manova(X, group_labels, group_colors, group_markers=None, min_n=5,
                   title="", outstem=None, figsize=(9.0, 7.0), fontsize=40,
                   use_shrinkage=False, show_legend=True, show_figure=False, point_size=8, legend_marker_size=20):
    """Fit LDA on X against group_labels, plot LD1 vs LD2, run MANOVA + pairwise MANOVA (FDR).

    X             : (n, d) feature matrix -- already reduced to a fixed number of PCs, any
                    representation (sparse landmarks, dense correspondences, latents).
    group_labels  : (n,) array-like of group names. NaN/None rows are dropped.
    group_colors  : dict {group_name: (r,g,b) or hex}.
    group_markers : optional dict {group_name: matplotlib marker char}; default 'o'.
    min_n         : groups with fewer than this many specimens are dropped -- LDA/MANOVA are
                    unstable or meaningless on a handful of points (e.g. the 'snake' category).
    title         : prefix only; the PC count and accuracy get appended automatically.

    Returns the fitted lda, LD scores, MANOVA tables, and cross-validated balanced accuracy.
    """
    group_labels = pd.Series(group_labels).reset_index(drop=True)
    keep = group_labels.notna().values
    X, group_labels = np.asarray(X)[keep], group_labels[keep].reset_index(drop=True)

    counts = group_labels.value_counts()
    small = counts[counts < min_n].index.tolist()
    if small:
        print(f"\033[33mDropping {len(small)} group(s) with < {min_n} specimens: {small}\033[0m")
        keep2 = ~group_labels.isin(small)
        X, group_labels = X[keep2.values], group_labels[keep2].reset_index(drop=True)

    y = group_labels.values
    levels = sorted(set(y))
    counts = group_labels.value_counts()
    if len(levels) < 2:
        raise ValueError(f"Only {len(levels)} group(s) left after filtering -- need at least 2")

    group_markers = group_markers or {}

    lda = (LDA(solver="eigen", shrinkage="auto", n_components=min(len(levels) - 1, X.shape[1]))
           if use_shrinkage else
           LDA(solver="svd", n_components=min(len(levels) - 1, X.shape[1])))
    X_lda = lda.fit_transform(X, y)
    explained = getattr(lda, "explained_variance_ratio_", None)

    n_splits = min(5, counts.reindex(levels).min())
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    clf_cv = LDA(solver="eigen", shrinkage="auto") if use_shrinkage else LDA(solver="svd")
    cv_scores = cross_val_score(clf_cv, X, y, cv=cv,
                                scoring=make_scorer(balanced_accuracy_score))

    chance = 1 / len(levels)
    print(f"{title}: cross-validated balanced accuracy = {cv_scores.mean():.3f} \u00b1 {cv_scores.std():.3f}  "
          f"(chance = {chance:.3f}, {len(levels)} groups, n={len(y)})")

    fig, ax = plt.subplots(figsize=figsize)
    for lev in levels:
        idx = y == lev
        ax.scatter(X_lda[idx, 0], X_lda[idx, 1] if X_lda.shape[1] > 1 else np.zeros(idx.sum()),
                  marker=group_markers.get(lev, "o"), color=group_colors.get(lev, (0.5, 0.5, 0.5)),
                  s=point_size, linewidths=0, label=lev.upper(), rasterized=True)
    ax.set_xlabel("LD1", fontsize=fontsize)
    ax.set_ylabel("LD2", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize - 4)
    ax.set_title(f"{title} {X.shape[1]}PCs\nACC = {cv_scores.mean():.3f}", fontsize=fontsize)
    ax.grid(alpha=0.25, lw=0.5)
    ax.set_axisbelow(True)

    if show_legend:
        handles = [plt.Line2D([0], [0], marker=group_markers.get(lev, "o"), linestyle="None",
                              markerfacecolor=group_colors.get(lev, (0.5, 0.5, 0.5)),
                              markeredgecolor=group_colors.get(lev, (0.5, 0.5, 0.5)),
                              markersize=legend_marker_size / 2, label=lev.upper())
                   for lev in levels]
        ax.legend(handles=handles, fontsize=fontsize - 1, frameon=False,
                  loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3)

    fig.subplots_adjust(bottom=0.12, left=0.12, right=0.95, top=0.85)
    if outstem:
        fig.savefig(str(OUT_DIR / f"{outstem}_lda.png"), dpi=300)
    if show_figure:
        plt.show()
    plt.close(fig)

    df_lda = pd.DataFrame(X_lda, columns=[f"LD{i+1}" for i in range(X_lda.shape[1])])
    df_lda["group"] = y
    formula = " + ".join([c for c in df_lda.columns if c != "group"])
    manova = MANOVA.from_formula(f"{formula} ~ group", data=df_lda)
    mv_result = manova.mv_test()

    n_features = X_lda.shape[1]
    pairwise_results = []
    for g1, g2 in combinations(levels, 2):
        df_pair = df_lda[df_lda["group"].isin([g1, g2])]
        pcounts = df_pair["group"].value_counts()
        if pcounts.min() <= n_features:
            pairwise_results.append({"Group 1": g1, "Group 2": g2, "Wilks' lambda": np.nan,
                                     "p-value": np.nan, "note": "underpowered"})
            continue
        stats_df = MANOVA.from_formula(f"{formula} ~ group", data=df_pair).mv_test().results["group"]["stat"]
        wilks_row = stats_df.loc["Wilks' lambda"]
        pairwise_results.append({"Group 1": g1, "Group 2": g2,
                                 "Wilks' lambda": wilks_row["Value"], "Num DF": wilks_row["Num DF"],
                                 "Den DF": wilks_row["Den DF"], "F Value": wilks_row["F Value"],
                                 "p-value": wilks_row["Pr > F"], "note": ""})

    pairwise_df = pd.DataFrame(pairwise_results)
    valid = pairwise_df["p-value"].notna()
    if valid.any():
        reject, p_corr, *_ = multipletests(pairwise_df.loc[valid, "p-value"].values, method="fdr_bh")
        pairwise_df["p-value (FDR corrected)"] = np.nan
        pairwise_df["significant"] = False
        pairwise_df.loc[valid, "p-value (FDR corrected)"] = p_corr
        pairwise_df.loc[valid, "significant"] = reject
    pairwise_df = pairwise_df.sort_values("p-value")

    if outstem:
        pairwise_df.to_csv(OUT_DIR / f"{outstem}_manova_pairwise.csv", index=False)

    return {"lda": lda, "X_lda": X_lda, "labels": y, "explained_variance": explained,
            "cv_balanced_accuracy": cv_scores.mean(), "cv_std": cv_scores.std(), "chance": chance,
            "mv_result": mv_result, "pairwise_df": pairwise_df}

# LDA at various % var retained thresholds from PC scores
def lda_threshold_grid(reps_by_threshold, labels, group_colors, thresholds, col_order, col_titles,
                       row_labels=None, min_n=10, use_shrinkage=True, show_accuracy=False,
                       width=2100, height=2100, font_size=45, marker_size=12,
                       legend_marker_size=60, outstem=None):
    labels = pd.Series(labels).reset_index(drop=True)
    nr, nc = len(thresholds), len(col_order)
    fig = make_subplots(rows=nr, cols=nc, horizontal_spacing=0.055, vertical_spacing=0.11)

    seen, subtitles = set(), {}
    for r, thr in enumerate(thresholds, start=1):
        for c, rep in enumerate(col_order, start=1):
            X = np.asarray(reps_by_threshold[(thr, rep)])
            keep = labels.notna().values
            Xs, ys = X[keep], labels[keep].reset_index(drop=True)
            small = ys.value_counts()[lambda v: v < min_n].index
            k2 = ~ys.isin(small)
            Xs, ys = Xs[k2.values], ys[k2].reset_index(drop=True).values

            levels = sorted(set(ys))
            clf = (LDA(solver="eigen", shrinkage="auto",
                       n_components=min(len(levels) - 1, Xs.shape[1]))
                   if use_shrinkage else LDA(solver="svd"))
            X_lda = clf.fit_transform(Xs, ys)

            acc = np.nan
            if show_accuracy:
                nsp = min(5, pd.Series(ys).value_counts().min())
                cv = StratifiedKFold(n_splits=nsp, shuffle=True, random_state=42)
                clf_cv = LDA(solver="eigen", shrinkage="auto") if use_shrinkage else LDA(solver="svd")
                acc = cross_val_score(clf_cv, Xs, ys, cv=cv,
                                      scoring=make_scorer(balanced_accuracy_score)).mean()
            subtitles[(r, c)] = f"{Xs.shape[1]} PCs" + (f"<br>ACC = {acc:.3f}" if show_accuracy else "")

            for lev in levels:
                idx = ys == lev
                seen.add(lev)
                fig.add_trace(go.Scatter(
                    x=X_lda[idx, 0], y=X_lda[idx, 1], mode="markers",
                    name=str(lev).upper(), legendgroup=str(lev), showlegend=False,
                    marker=dict(color=plotly_color(group_colors.get(lev, (.5, .5, .5))),
                                size=marker_size, symbol="circle"),
                    hovertemplate=f"{lev}<extra></extra>"), row=r, col=c)
            fig.update_xaxes(title_text="LD1", row=r, col=c)
            fig.update_yaxes(title_text="LD2", row=r, col=c)

    fig.update_layout(width=width, height=height, plot_bgcolor="white",
                      font=dict(size=font_size),
                      legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.05,
                                  yanchor="top", font=dict(size=font_size)),
                      margin=dict(t=170, b=200, l=170, r=60))
    fig.update_xaxes(showline=True, linewidth=2, linecolor="black", mirror=True,
                     ticks="outside", showticklabels=False, showgrid=False)
    fig.update_yaxes(showline=True, linewidth=2, linecolor="black", mirror=True,
                     ticks="outside", showticklabels=False, showgrid=False)

    anns = []
    for c, ct in enumerate(col_titles, start=1):
        xr = fig.get_subplot(1, c).xaxis.domain
        anns.append(dict(x=(xr[0] + xr[1]) / 2, y=1.1, xref="paper", yref="paper",
                         text=ct, showarrow=False, xanchor="center",
                         font=dict(size=font_size * 1.15)))
    row_labels = row_labels or [f"{100*t:.0f}% VAR" for t in thresholds]
    for r, rl in enumerate(row_labels, start=1):
        yr = fig.get_subplot(r, 1).yaxis.domain
        anns.append(dict(x=-0.075, y=(yr[0] + yr[1]) / 2, xref="paper", yref="paper",
                         text=rl, showarrow=False, textangle=-90, yanchor="middle",
                         font=dict(size=font_size * 1.15)))
    for (r, c), st in subtitles.items():
        xr = fig.get_subplot(r, c).xaxis.domain
        yr = fig.get_subplot(r, c).yaxis.domain
        anns.append(dict(x=(xr[0] + xr[1]) / 2, y=yr[1] + 0.004, xref="paper", yref="paper",
                         text=st, showarrow=False, xanchor="center", yanchor="bottom",
                         font=dict(size=font_size * 0.9)))
    fig.update_layout(annotations=anns)

    for lev in sorted(seen):
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
                                 name=str(lev).upper(), legendgroup=str(lev), showlegend=True,
                                 marker=dict(color=plotly_color(group_colors.get(lev, (.5, .5, .5))),
                                             size=legend_marker_size, symbol="circle")))
    if outstem:
        fig.write_html(str(OUT_DIR / f"{outstem}.html"), include_plotlyjs="cdn")
        fig.write_image(str(OUT_DIR / f"{outstem}.png"))
    fig.show()
    return fig

def manova_stats(X_lda, ys):
    """Overall MANOVA on the LD scores. Returns Wilks' lambda, F, p, and partial eta-squared."""
    df = pd.DataFrame(X_lda, columns=[f"LD{i+1}" for i in range(X_lda.shape[1])])
    df["group"] = ys
    formula = " + ".join(c for c in df.columns if c != "group")
    stat = MANOVA.from_formula(f"{formula} ~ group", data=df).mv_test().results["group"]["stat"]
    wilks = stat.loc["Wilks' lambda"]
    lam = float(wilks["Value"])
    return {"wilks_lambda": lam,
            "manova_F": float(wilks["F Value"]),
            "manova_num_df": float(wilks["Num DF"]),
            "manova_den_df": float(wilks["Den DF"]),
            "manova_p": float(wilks["Pr > F"]),
            "partial_eta_sq": 1 - lam ** (1 / min(X_lda.shape[1], len(set(ys)) - 1))}

def summarise_representation(X, labels, groups, rep_name, label_set, threshold, n_pcs):
    labels = pd.Series(labels).reset_index(drop=True)
    groups = pd.Series(groups).reset_index(drop=True)
    keep = labels.notna().values & groups.notna().values
    Xs = np.asarray(X)[keep]
    ys = labels[keep].reset_index(drop=True)
    gs = groups[keep].reset_index(drop=True)

    # filter on number of distinct specimens, not number of vertebrae -- a class carried by
    # two animals cannot be split across folds under grouped CV no matter how many vertebrae it has
    per_class_specimens = pd.DataFrame({"g": gs, "y": ys}).drop_duplicates()["y"].value_counts()
    small = per_class_specimens[per_class_specimens < MIN_SPECIMENS].index.tolist()
    if small:
        k2 = ~ys.isin(small)
        Xs, ys, gs = Xs[k2.values], ys[k2].reset_index(drop=True), gs[k2].reset_index(drop=True)
    ys, gs = ys.values, gs.values

    lev = sorted(set(ys))
    chance = 1 / len(lev)
    clf = lambda: LDA(solver="eigen", shrinkage="auto")

    # random-split CV + permutation null
    cv_r = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    acc_r = cross_val_score(clf(), Xs, ys, cv=cv_r, scoring=make_scorer(balanced_accuracy_score))
    score_p, perm_scores, pval = permutation_test_score(
        clf(), Xs, ys, cv=cv_r, n_permutations=N_PERM,
        scoring=make_scorer(balanced_accuracy_score), random_state=42, n_jobs=-1)

    # grouped (across-individual) CV
    n_sp_min = pd.DataFrame({"g": gs, "y": ys}).drop_duplicates()["y"].value_counts().min()
    n_splits_g = int(min(5, n_sp_min))
    cv_g = StratifiedGroupKFold(n_splits=n_splits_g, shuffle=True, random_state=42)
    acc_g = cross_val_score(clf(), Xs, ys, groups=gs, cv=cv_g,
                            scoring=make_scorer(balanced_accuracy_score))

    # per-class recall (random-split predictions)
    y_pred = cross_val_predict(clf(), Xs, ys, cv=cv_r)
    rec = recall_score(ys, y_pred, labels=lev, average=None, zero_division=0)

    row = {"representation": rep_name, "label_set": label_set, "threshold": threshold,
           "n_pcs": n_pcs, "n_specimens_total": len(ys), "n_classes": len(lev),
           "chance_level": chance,
           "cv_random_balanced_acc": acc_r.mean(), "cv_random_sd": acc_r.std(),
           "cv_grouped_balanced_acc": acc_g.mean(), "cv_grouped_sd": acc_g.std(),
           "cv_grouped_n_splits": n_splits_g,
           "perm_null_mean": perm_scores.mean(), "perm_p_value": pval,
           "classes_dropped": ";".join(small) if small else ""}
    row.update({f"recall_{c}": r for c, r in zip(lev, rec)})

    # MANOVA
    lda_fit = clf().fit(Xs, ys)
    X_lda = lda_fit.transform(Xs)
    row.update(manova_stats(X_lda, ys))

    return row

6 traits: ['arboreal', 'burrowing', 'grass-swimmer', 'saxicolous', 'snake', 'terrestrial']


### Load the three shape representations
Sparse landmarks and dense correspondences are both GPA-aligned/scaled already, so `gm_prcomp`
(Procrustes PCA, from `NSM.morphometrics`) runs directly on them. Latent codes get plain `sklearn`
PCA since they aren't Procrustes shape coordinates.

In [5]:
# 28 sparse landmarks
sparse_coords = np.stack([load_mrk_json(SPARSE_LM_DIR / (os.path.splitext(f)[0] + ".mrk.json"))[0]
                          for f in all_vtk_files])
sparse_mean, _ = load_mrk_json(SPARSE_MEAN_FN)
print(f"Sparse landmarks: {sparse_coords.shape}")
assert sparse_mean.shape == sparse_coords.shape[1:], "sparse atlas / specimen landmark count mismatch"

pca_sparse = gm_prcomp(sparse_coords)
print(f"{pca_sparse['x'].shape[1]} non-trivial PCs from {sparse_coords.shape[1]*3} sparse coordinates")

# Dense correspondences 
dense_coords = np.stack([load_mrk_json(DENSE_LM_DIR / (os.path.splitext(f)[0] + ".mrk.json"))[0]
                         for f in all_vtk_files])
dense_mean, _ = load_mrk_json(DENSE_MEAN_FN)
print(f"Dense correspondences: {dense_coords.shape}")
assert dense_mean.shape == dense_coords.shape[1:], "dense atlas / specimen point count mismatch"

pca_dense = gm_prcomp(dense_coords)
print(f"{pca_dense['x'].shape[1]} non-trivial PCs from {dense_coords.shape[1]*3} dense coordinates")

# NSM latent codes
pca_latent_model = PCA(n_components=5)
latent_scores = pca_latent_model.fit_transform(codes)
latent_prop = pca_latent_model.explained_variance_ratio_
print(f"Latent PCA: {latent_scores.shape[1]} components, "
      f"PC1+PC2 = {100*(latent_prop[0]+latent_prop[1]):.1f}% of variance")

Sparse landmarks: (2316, 28, 3)
84 non-trivial PCs from 84 sparse coordinates
Dense correspondences: (2316, 4943, 3)
2315 non-trivial PCs from 14829 dense coordinates
Latent PCA: 5 components, PC1+PC2 = 20.2% of variance


## LDA / KDE / MANOVA: is group structure meaningful, across representations and label sets?

### Run across all three representations, colored by trait

`N_PC_MATCH` fixes the number of retained PCs the same way across sparse, dense, and latents, so
none of the three gets a separability advantage purely from having more dimensions to work with.

In [8]:
# Build each representation once at full dimensionality, plus its cumulative-variance 

# Plot params
DPI = 300
PANEL_W_PX, PANEL_H_PX = 700, 650
LDA_FIGSIZE = (PANEL_W_PX / DPI, PANEL_H_PX / DPI)     # (2.333, 2.167) inches
PX_TO_PT = 72 / DPI                                     # CSS px -> matplotlib points
LDA_FONTSIZE      = 30 * PX_TO_PT                       # 7.2 pt  == plotly size=30
LDA_POINT_S       = (6 * PX_TO_PT) ** 2                 # s=2.07  == plotly marker size=6
LDA_LEGEND_MARKER = 20 * PX_TO_PT                       # 4.8 pt  == plotly legend size=20

# PCA
pca_latent = PCA(n_components=None).fit(codes)
latent_full = pca_latent.transform(codes)
latent_cum = np.cumsum(pca_latent.explained_variance_ratio_)

# PCA dict
reps = {"Landmarks": (pca_sparse["x"], pca_sparse["cum"]),
        "Dense":     (pca_dense["x"],  pca_dense["cum"]),
        "Latents":   (latent_full,     latent_cum)}

# PCA retained variance thresholds for inspection
THRESHOLDS = [0.90, 0.95, 0.99, 0.997, 1.0]

curve_rows = []
for target in ["trait", "broad_taxon_for_plotting"]:
    labels = pd.Series(specimens[target].values)
    keep = labels.notna().values
    for name, (X, cum) in reps.items():
        for thr in THRESHOLDS:
            n = n_pcs_for_variance(cum, thr)
            Xs, ys = X[keep, :n], labels[keep].reset_index(drop=True)
            small = ys.value_counts()[lambda c: c < 10].index.tolist()
            k2 = ~ys.isin(small)
            Xs, ys = Xs[k2.values], ys[k2].values
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            scores = cross_val_score(LDA(solver="eigen", shrinkage="auto"), Xs, ys, cv=cv,
                                     scoring=make_scorer(balanced_accuracy_score))
            curve_rows.append({"target": target, "representation": name, "threshold": thr,
                               "n_pcs": n, "accuracy": scores.mean(), "sd": scores.std()})

curve = pd.DataFrame(curve_rows)
for target in curve["target"].unique():
    sub = curve[curve["target"] == target]
    combined = sub.copy()
    combined["value"] = (combined["accuracy"].round(3).astype(str) + " ±" +
                         combined["sd"].round(3).astype(str) + " (n=" +
                         combined["n_pcs"].astype(str) + ")")
    out = (combined.pivot(index="threshold", columns="representation", values="value")
                   .reindex(columns=["Landmarks", "Dense", "Latents"]))
    print(f"\n=== {target} ===")
    print(out.to_string())


=== trait ===
representation            Landmarks                 Dense               Latents
threshold                                                                      
0.900            0.68 ±0.027 (n=16)   0.674 ±0.031 (n=12)  0.901 ±0.017 (n=266)
0.950           0.716 ±0.024 (n=27)   0.738 ±0.036 (n=30)  0.894 ±0.017 (n=358)
0.990           0.748 ±0.029 (n=55)  0.927 ±0.012 (n=315)  0.883 ±0.021 (n=466)
0.997            0.76 ±0.029 (n=68)   0.77 ±0.022 (n=967)   0.88 ±0.017 (n=496)
1.000           0.764 ±0.026 (n=84)   0.2 ±0.001 (n=2315)  0.871 ±0.017 (n=512)

=== broad_taxon_for_plotting ===
representation            Landmarks                  Dense               Latents
threshold                                                                       
0.900           0.785 ±0.018 (n=16)    0.661 ±0.008 (n=12)  0.965 ±0.015 (n=266)
0.950           0.858 ±0.011 (n=27)    0.893 ±0.005 (n=30)   0.967 ±0.01 (n=358)
0.990           0.917 ±0.012 (n=55)   0.988 ±0.008 (n=315)  0.956 ±

### Run across all three representations, colored by broad taxonomic clade

In [9]:
# LDA 3x3 grid: rows = variance thresholds, cols = representations. One plotly figure,
# same geometry/fonts/marker conventions as the PCA/t-SNE/UMAP panel.

THRESHOLDS = [0.90, 0.95, 0.99]
LABEL_COL, LABEL_TAG = "trait", "trait"
cols = ["Landmarks", "Dense", "Latents"]
col_titles = [f"Landmarks (N={sparse_coords.shape[1]})",
              f"Dense correspondences (N={dense_coords.shape[1]})",
              "NSM latents"]

reps_by_threshold = {}
for thr in THRESHOLDS:
    reps_by_threshold[(thr, "Landmarks")] = pca_sparse["x"][:, :n_pcs_for_variance(pca_sparse["cum"], thr)]
    reps_by_threshold[(thr, "Dense")]     = pca_dense["x"][:, :n_pcs_for_variance(pca_dense["cum"], thr)]
    reps_by_threshold[(thr, "Latents")]   = latent_full[:, :n_pcs_for_variance(latent_cum, thr)]

_ = lda_threshold_grid(reps_by_threshold, specimens[LABEL_COL].values, trait_colors,
                       THRESHOLDS, cols, col_titles,
                       outstem=f"{RUN}_lda_{LABEL_TAG}_3x3_thresholds")

---
## Validation: does each representation better separate taxonomy, trait, or both?

Pulls the cross-validated LDA accuracy from every (representation x label-set) run above into one
summary table -- the direct, quantitative answer to the taxonomy-vs-trait hypothesis, rather than
comparing plots by eye.

In [ ]:
# Summary statistics for the paper: one tidy CSV covering every
# (representation x label set x variance threshold) combination.

THRESHOLDS_REPORT = [0.90, 0.95, 0.99, 0.997, 1.0]
N_PERM = 200
MIN_SPECIMENS = 5          # drop a class with fewer than this many distinct ANIMALS

rows = []
for label_set, colour_map in [("trait", trait_colors),
                              ("broad_taxon_for_plotting", family_base_colors)]:
    for thr in THRESHOLDS_REPORT:
        n_s = n_pcs_for_variance(pca_sparse["cum"], thr)
        n_d = n_pcs_for_variance(pca_dense["cum"], thr)
        n_l = n_pcs_for_variance(latent_cum, thr)
        for rep_name, X, n_pc in [("Landmarks", pca_sparse["x"][:, :n_s], n_s),
                                  ("Dense",     pca_dense["x"][:, :n_d],  n_d),
                                  ("Latents",   latent_full[:, :n_l],     n_l)]:
            rows.append(summarise_representation(
                X, specimens[label_set].values, specimens["specimen_id"].values,
                rep_name, label_set, thr, n_pc))
            print(f"  done: {label_set} / {thr} / {rep_name}")

summary = pd.DataFrame(rows)
outpath = OUT_DIR / f"{RUN}_lda_summary_statistics.csv"
summary.to_csv(outpath, index=False)
print(f"\nWrote {outpath.resolve()}  ({len(summary)} rows)")

# Compact view for the manuscript table
for label_set in summary["label_set"].unique():
    sub = summary[summary["label_set"] == label_set]
    disp = sub.assign(
        random=lambda d: d["cv_random_balanced_acc"].map("{:.3f}".format) + " ±"
                         + d["cv_random_sd"].map("{:.3f}".format),
        grouped=lambda d: d["cv_grouped_balanced_acc"].map("{:.3f}".format) + " ±"
                          + d["cv_grouped_sd"].map("{:.3f}".format))
    print(f"\n=== {label_set} (chance = {sub['chance_level'].iloc[0]:.3f}) ===")
    print(disp.pivot(index="threshold", columns="representation", values="random")
              .reindex(columns=["Landmarks", "Dense", "Latents"]).to_string())
    print("\ngrouped (across-individual) CV:")
    print(disp.pivot(index="threshold", columns="representation", values="grouped")
              .reindex(columns=["Landmarks", "Dense", "Latents"]).to_string())
    print("\nn_pcs:")
    print(sub.pivot(index="threshold", columns="representation", values="n_pcs")
             .reindex(columns=["Landmarks", "Dense", "Latents"]).to_string())

y_pred contains classes not in y_true


  done: trait / 0.9 / Landmarks


y_pred contains classes not in y_true


  done: trait / 0.9 / Dense


y_pred contains classes not in y_true


...

  done: trait / 0.997 / Landmarks
  done: trait / 0.997 / Dense


y_pred contains classes not in y_true


  done: trait / 0.997 / Latents


y_pred contains classes not in y_true


  done: trait / 1.0 / Landmarks
